# TripleMRNet-inspired MRNet: Three independent pretrained AlexNets

This notebook implements a **three-plane feature-fusion model** inspired by the TripleMRNet paper, and applies it to **all three labels** in your MRNet dataset:

- **Abnormality** (`abnormal`)
- **ACL tear** (`acl`)
- **Meniscal tear** (`meniscus`)

For each task, a separate binary model is trained using sagittal, coronal and axial MRI volumes. Each model has three independent ImageNet-pretrained AlexNet encoders (one per plane), and concatenates their feature embeddings before binary classification.

**Important distinction:** this creates three separate binary models, not one shared model with three output neurons. The paper excerpt describes ACL and meniscal tear tasks; abnormality is added here so that the experiment also covers the third label in your dataset.

**Adaptation notes:** The paper's text is contradictory about ImageNet initialization versus training from scratch. This notebook follows the pretrained-AlexNet variant. The paper does not fully specify every tensor operation, so this is a practical TripleMRNet-inspired implementation, not a bit-for-bit reproduction.

It reuses your cached `.npy` volumes and exam parquet files; no Redivis download is performed.


## 1. Mount Drive and imports

Expected table files:
- `MyDrive/mrnet/train_exams.parquet`
- `MyDrive/mrnet/valid_exams.parquet`

Expected cache layout (the first layout is preferred):
- `cache/train/sagittal/<exam_id>.npy`
- `cache/train/coronal/<exam_id>.npy`
- `cache/train/axial/<exam_id>.npy`
- corresponding `cache/valid/...`

The loader also checks `cache/<split>/<exam_id>/<view>.npy`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import copy, random, time, gc
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import alexnet, AlexNet_Weights

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, average_precision_score
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Mounted at /content/drive
Device: cuda
GPU: Tesla T4


## 2. Paths and training configuration

Start with the settings below. Three independent AlexNets are larger than the shared ResNet18 model, so the exam batch size is deliberately small. The encoder is fine-tuned with a lower learning rate than the new classifier head.

In [ ]:
DRIVE_ROOT = Path("/content/drive/MyDrive/mrnet")
CACHE_ROOT = DRIVE_ROOT / "cache"
TRAIN_TABLE = DRIVE_ROOT / "train_exams.parquet"
VALID_TABLE = DRIVE_ROOT / "valid_exams.parquet"

ID_COL = "exam_id"
ABNORMAL_COL = "abnormal"
ACL_COL = "acl"
MENISCUS_COL = "meniscus"
TARGET_COLS = [ABNORMAL_COL, ACL_COL, MENISCUS_COL]
VIEWS = ["sagittal", "coronal", "axial"]

EPOCHS = 10                 # feasibility starting point; increase toward 50 if runtime allows
BATCH_SIZE = 1              # variable slice counts + three encoders; safest starting point
SLICE_BATCH_SIZE = 16       # slices processed per AlexNet forward pass
NUM_WORKERS = 0
IMAGE_SIZE = 224
MAX_SLICES_PER_VIEW = None  # None = use every slice; set e.g. 32 only for a quick smoke test

BACKBONE_LR = 1e-5
HEAD_LR = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.5

SAVE_CHECKPOINTS = True
CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints" / "triplemrnet_pretrained_alexnet"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

assert TRAIN_TABLE.exists(), f"Missing {TRAIN_TABLE}"
assert VALID_TABLE.exists(), f"Missing {VALID_TABLE}"
print("Cache:", CACHE_ROOT)


Cache: /content/drive/MyDrive/mrnet/cache


## 3. Load tables and check labels

We train **three independent binary tasks**: abnormality, ACL tear and meniscal tear. Each task gets its own three-plane model. This keeps the paper-inspired separate-task setup while including all three labels available in your MRNet tables.


In [ ]:
train_df = pd.read_parquet(TRAIN_TABLE)
valid_df = pd.read_parquet(VALID_TABLE)

required = [ID_COL] + TARGET_COLS
for col in required:
    if col not in train_df.columns or col not in valid_df.columns:
        raise ValueError(f"Missing {col}. Train columns: {list(train_df.columns)}")

def normalize_exam_ids(df):
    df = df.copy()
    df[ID_COL] = df[ID_COL].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(4)
    return df

train_df = normalize_exam_ids(train_df)
valid_df = normalize_exam_ids(valid_df)

overlap = set(train_df[ID_COL]) & set(valid_df[ID_COL])
print("Train:", len(train_df), "Validation:", len(valid_df))
print("Train/validation ID overlap:", len(overlap))
if overlap:
    print("WARNING: overlapping IDs:", sorted(overlap)[:10])

for col in TARGET_COLS:
    print(f"\n{col}")
    print("Train:", train_df[col].value_counts(dropna=False).to_dict())
    print("Valid:", valid_df[col].value_counts(dropna=False).to_dict())


Train: 1129 Validation: 119
Train/validation ID overlap: 0

abnormal
Train: {1.0: 912, 0.0: 217}
Valid: {1.0: 95, 0.0: 24}

acl
Train: {0.0: 921, 1.0: 208}
Valid: {0.0: 65, 1.0: 54}

meniscus
Train: {0.0: 732, 1.0: 397}
Valid: {0.0: 67, 1.0: 52}


## 4. Locate and inspect cached volumes

This supports the cache layouts used in the earlier MRNet work. If a file is not found, stop here and adjust the path candidates to match your actual Drive folders.

In [ ]:
def find_volume_path(split, view, exam_id):
    candidates = [
        CACHE_ROOT / split / view / f"{exam_id}.npy",
        CACHE_ROOT / split / exam_id / f"{view}.npy",
        CACHE_ROOT / split / f"{view}_{exam_id}.npy",
        CACHE_ROOT / split / f"{exam_id}_{view}.npy",
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(
        f"Missing {view} volume for exam {exam_id} ({split}). Checked:\\n" +
        "\\n".join(map(str, candidates))
    )

def inspect_cache(df, split, n=3):
    for exam_id in df[ID_COL].head(n):
        for view in VIEWS:
            p = find_volume_path(split, view, exam_id)
            arr = np.load(p, mmap_mode="r")
            print(exam_id, view, arr.shape, arr.dtype, p)

inspect_cache(train_df, "train", min(3, len(train_df)))
inspect_cache(valid_df, "valid", min(3, len(valid_df)))

0060 sagittal (18, 256, 256) uint8 /content/drive/MyDrive/mrnet/cache/train/sagittal/0060.npy
0060 coronal (28, 256, 256) uint8 /content/drive/MyDrive/mrnet/cache/train/coronal/0060.npy
0060 axial (39, 256, 256) uint8 /content/drive/MyDrive/mrnet/cache/train/axial/0060.npy
0027 sagittal (19, 256, 256) uint8 /content/drive/MyDrive/mrnet/cache/train/sagittal/0027.npy
0027 coronal (28, 256, 256) uint8 /content/drive/MyDrive/mrnet/cache/train/coronal/0027.npy
0027 axial (22, 256, 256) uint8 /content/drive/MyDrive/mrnet/cache/train/axial/0027.npy
0053 sagittal (19, 256, 256) uint8 /content/drive/MyDrive/mrnet/cache/train/sagittal/0053.npy
0053 coronal (18, 256, 256) uint8 /content/drive/MyDrive/mrnet/cache/train/coronal/0053.npy
0053 axial (19, 256, 256) uint8 /content/drive/MyDrive/mrnet/cache/train/axial/0053.npy
1185 sagittal (21, 256, 256) uint8 /content/drive/MyDrive/mrnet/cache/valid/sagittal/1185.npy
1185 coronal (17, 256, 256) uint8 /content/drive/MyDrive/mrnet/cache/valid/coronal/1

## 5. Dataset and variable-length batch collation

Each exam has three 3D volumes. Since the number of slices can vary between exams, the collate function keeps each exam's volumes as a list instead of padding them into one large tensor.

In [ ]:
class ThreePlaneMRNetDataset(Dataset):
    def __init__(self, df, split, target_col):
        self.df = df.dropna(subset=[target_col]).copy().reset_index(drop=True)
        self.split = split
        self.target_col = target_col
        self.df[target_col] = self.df[target_col].astype(np.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        exam_id = str(row[ID_COL])
        volumes = []
        for view in VIEWS:
            arr = np.load(find_volume_path(self.split, view, exam_id)).astype(np.float32)
            if arr.ndim != 3:
                raise ValueError(f"{exam_id} {view}: expected (slices,H,W), got {arr.shape}")
            # MRNet cached volumes are uint8 intensity arrays; convert to [0,1].
            if arr.max() > 1.0:
                arr /= 255.0
            volumes.append(torch.from_numpy(arr))
        label = torch.tensor(float(row[self.target_col]), dtype=torch.float32)
        return volumes[0], volumes[1], volumes[2], label, exam_id

def collate_exams(batch):
    s, c, a, y, ids = zip(*batch)
    return list(s), list(c), list(a), torch.stack(y), list(ids)

def make_loader(df, split, target_col, shuffle):
    ds = ThreePlaneMRNetDataset(df, split, target_col)
    return DataLoader(
        ds, batch_size=BATCH_SIZE, shuffle=shuffle,
        num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"),
        collate_fn=collate_exams
    )

## 6. Independent AlexNet plane encoder

Each plane gets its **own AlexNet instance**. The models start from the same ImageNet-pretrained initialization, but their parameters are separate objects and are updated independently.

For every slice:
1. Expand grayscale to three identical channels.
2. Resize to 224 × 224.
3. Apply ImageNet normalization.
4. Run AlexNet's convolutional feature extractor.
5. Use adaptive average pooling to produce a 256-dimensional vector per slice.
6. Average the slice vectors to produce one 256-dimensional embedding for that plane.

AlexNet's original 1,000-class classifier is not used.

**Device-handling fix:** MRI volumes are moved to the encoder's device inside `forward()`, so CPU DataLoader batches and CUDA model buffers do not cause a device mismatch.

In [ ]:
class AlexNetPlaneEncoder(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights = AlexNet_Weights.DEFAULT if pretrained else None
        net = alexnet(weights=weights)
        self.features = net.features
        self.avgpool = net.avgpool

        self.register_buffer(
            "mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        )
        self.register_buffer(
            "std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        )

    def forward(self, volume):
        # Keep the MRI volume on the same device as this encoder's buffers/model.
        # The DataLoader supplies CPU tensors, while the model may be on CUDA.
        volume = volume.to(self.mean.device, non_blocking=True)

        # volume: (S,H,W), already scaled approximately to [0,1]
        if MAX_SLICES_PER_VIEW is not None and volume.shape[0] > MAX_SLICES_PER_VIEW:
            # Deterministic, evenly spaced slice selection for smoke tests only.
            idx = torch.linspace(
                0, volume.shape[0] - 1, MAX_SLICES_PER_VIEW
            ).long()
            volume = volume[idx]

        x = volume.unsqueeze(1).repeat(1, 3, 1, 1)  # (S,3,H,W)
        x = F.interpolate(
            x, size=(IMAGE_SIZE, IMAGE_SIZE),
            mode="bilinear", align_corners=False
        )
        x = (x - self.mean) / self.std

        per_slice_features = []
        for start in range(0, x.shape[0], SLICE_BATCH_SIZE):
            xb = x[start:start + SLICE_BATCH_SIZE]
            feature_map = self.features(xb)       # (B,256,6,6) for 224px input
            pooled_map = self.avgpool(feature_map) # AlexNet adaptive pool to (B,256,6,6)
            vector = F.adaptive_avg_pool2d(pooled_map, 1).flatten(1) # (B,256)
            per_slice_features.append(vector)

        per_slice_features = torch.cat(per_slice_features, dim=0)  # (S,256)
        # Aggregate all slices into one plane-level representation.
        plane_embedding = F.adaptive_avg_pool1d(
            per_slice_features.transpose(0, 1).unsqueeze(0), 1
        ).squeeze(0).squeeze(-1)  # (256,)
        return plane_embedding

## 7. Three-plane feature fusion and task classifier

The three 256-dimensional plane embeddings are concatenated into a 768-dimensional vector. A fully connected layer produces one logit for the current binary task.

This is **feature-level fusion**, not probability-level fusion and not attention-based fusion.

In [ ]:
class TripleMRNetThreePlane(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.sagittal_encoder = AlexNetPlaneEncoder(pretrained)
        self.coronal_encoder = AlexNetPlaneEncoder(pretrained)
        self.axial_encoder = AlexNetPlaneEncoder(pretrained)

        self.classifier = nn.Sequential(
            nn.Dropout(DROPOUT),
            nn.Linear(256 * 3, 1)
        )

    def forward(self, sagittal_volumes, coronal_volumes, axial_volumes):
        fused_exam_features = []
        for sag, cor, axi in zip(sagittal_volumes, coronal_volumes, axial_volumes):
            fs = self.sagittal_encoder(sag)
            fc = self.coronal_encoder(cor)
            fa = self.axial_encoder(axi)
            fused_exam_features.append(torch.cat([fs, fc, fa], dim=0))

        fused_batch = torch.stack(fused_exam_features, dim=0)  # (B,768)
        return self.classifier(fused_batch).squeeze(1)

# Quick architecture check: all three encoders must be separate objects.
_check = TripleMRNetThreePlane(pretrained=True)
print("Sagittal and coronal are same object:", _check.sagittal_encoder is _check.coronal_encoder)
print("Coronal and axial are same object:", _check.coronal_encoder is _check.axial_encoder)
print("Trainable parameters:", f"{sum(p.numel() for p in _check.parameters() if p.requires_grad):,}")
del _check
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 146MB/s]


Sagittal and coronal are same object: False
Coronal and axial are same object: False
Trainable parameters: 7,409,857


## 8. Loss, metrics and training helpers

The paper describes inverse-prevalence weighting. Here we use `BCEWithLogitsLoss(pos_weight=negative_count/positive_count)`, a common positive-class weighting formulation. This is a practical imbalance correction, but is not guaranteed to be mathematically identical to the paper's per-example reciprocal-prevalence weighting.

In [ ]:
def make_pos_weight(df, target_col):
    labels = df.dropna(subset=[target_col])[target_col].astype(int)
    positives = int((labels == 1).sum())
    negatives = int((labels == 0).sum())
    if positives == 0 or negatives == 0:
        raise ValueError(f"{target_col}: need both positive and negative training examples")
    return torch.tensor([negatives / positives], dtype=torch.float32, device=device)

def compute_metrics(y_true, probabilities, threshold=0.5):
    y_pred = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) else 0.0,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn)
    }
    out["roc_auc"] = roc_auc_score(y_true, probabilities) if len(np.unique(y_true)) == 2 else np.nan
    out["average_precision"] = average_precision_score(y_true, probabilities) if len(np.unique(y_true)) == 2 else np.nan
    return out

def run_epoch(model, loader, criterion, optimizer=None):
    is_training = optimizer is not None
    model.train(is_training)
    total_loss = 0.0
    labels_all, probs_all, ids_all = [], [], []

    for s_vols, c_vols, a_vols, labels, exam_ids in tqdm(loader, leave=False):
        labels = labels.to(device)
        if is_training:
            optimizer.zero_grad(set_to_none=True)

        logits = model(s_vols, c_vols, a_vols)
        loss = criterion(logits, labels)

        if is_training:
            loss.backward()
            optimizer.step()

        probabilities = torch.sigmoid(logits).detach().cpu().numpy()
        total_loss += loss.item() * len(labels)
        labels_all.extend(labels.detach().cpu().numpy().astype(int).tolist())
        probs_all.extend(probabilities.tolist())
        ids_all.extend(exam_ids)

    mean_loss = total_loss / max(1, len(labels_all))
    return mean_loss, np.asarray(labels_all), np.asarray(probs_all), ids_all

## 10. Train one binary task at a time

The same architecture is initialized and trained independently for each label. Run the three task cells below in order. Each run saves its own best checkpoint, so the three tasks can be run separately if Colab disconnects.


In [ ]:
def train_task(target_col, task_name):
    train_loader = make_loader(train_df, "train", target_col, shuffle=True)
    valid_loader = make_loader(valid_df, "valid", target_col, shuffle=False)

    model = TripleMRNetThreePlane(pretrained=True).to(device)
    pos_weight = make_pos_weight(train_df, target_col)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    # Lower LR for pretrained convolutional encoders; higher LR for new classifier.
    optimizer = torch.optim.AdamW([
        {"params": model.sagittal_encoder.parameters(), "lr": BACKBONE_LR},
        {"params": model.coronal_encoder.parameters(), "lr": BACKBONE_LR},
        {"params": model.axial_encoder.parameters(), "lr": BACKBONE_LR},
        {"params": model.classifier.parameters(), "lr": HEAD_LR},
    ], weight_decay=WEIGHT_DECAY)

    best_val_loss = float("inf")
    best_state = None
    history = []

    for epoch in range(1, EPOCHS + 1):
        start = time.time()
        train_loss, _, _, _ = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, y_val, p_val, val_ids = run_epoch(model, valid_loader, criterion)
        metrics = compute_metrics(y_val, p_val, threshold=0.5)

        history.append({"epoch": epoch, "train_loss": train_loss,
                        "valid_loss": val_loss, **metrics,
                        "seconds": time.time() - start})
        print(
            f"{task_name} | epoch {epoch}/{EPOCHS} | "
            f"train loss {train_loss:.4f} | val loss {val_loss:.4f} | "
            f"F1 {metrics['f1']:.4f} | AUC {metrics['roc_auc']:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            if SAVE_CHECKPOINTS:
                torch.save({
                    "task": task_name,
                    "epoch": epoch,
                    "model_state_dict": best_state,
                    "valid_loss": best_val_loss,
                    "metrics_at_0_5": metrics,
                    "configuration": {
                        "pretrained_alexnet": True,
                        "independent_plane_encoders": True,
                        "views": VIEWS,
                        "epochs": EPOCHS,
                        "batch_size": BATCH_SIZE,
                    }
                }, CHECKPOINT_DIR / f"{task_name}_best.pt")

    if best_state is not None:
        model.load_state_dict(best_state)

    _, y_val, p_val, val_ids = run_epoch(model, valid_loader, criterion)
    final_metrics = compute_metrics(y_val, p_val, threshold=0.5)
    predictions = pd.DataFrame({
        "exam_id": val_ids,
        "actual": y_val,
        "probability": p_val,
        "predicted": (p_val >= 0.5).astype(int)
    })
    print(f"\\nBest {task_name} checkpoint metrics at threshold 0.5:")
    display(pd.Series(final_metrics))
    return model, pd.DataFrame(history), predictions, final_metrics

## 11. Train abnormality model

This third task is added because your dataset includes the `abnormal` label. It is not one of the two tasks described in the supplied TripleMRNet paper excerpt.


In [ ]:
abnormal_model, abnormal_history, abnormal_predictions, abnormal_metrics = train_task(ABNORMAL_COL, "abnormal")


  0%|          | 0/1129 [00:00<?, ?it/s]

  0%|          | 0/119 [00:00<?, ?it/s]

abnormal | epoch 1/10 | train loss 0.2472 | val loss 0.2077 | F1 0.9026 | AUC 0.8561


  0%|          | 0/1129 [00:00<?, ?it/s]

  0%|          | 0/119 [00:00<?, ?it/s]

abnormal | epoch 2/10 | train loss 0.1821 | val loss 0.1703 | F1 0.9091 | AUC 0.9162


... (23 outputs left out)

abnormal | epoch 10/10 | train loss 0.1067 | val loss 0.1911 | F1 0.9189 | AUC 0.9057


  0%|          | 0/119 [00:00<?, ?it/s]

\nBest abnormal checkpoint metrics at threshold 0.5:


,0
accuracy,0.865546
precision,0.975904
sensitivity,0.852632
specificity,0.916667
f1,0.910112
TP,81.000000
FP,2.000000
TN,22.000000
FN,14.000000
roc_auc,0.927193


## 12. Train ACL model

A fresh three-plane model is initialized for ACL tear detection.


In [ ]:
acl_model, acl_history, acl_predictions, acl_metrics = train_task(ACL_COL, "acl")


  0%|          | 0/1129 [00:00<?, ?it/s]

  0%|          | 0/119 [00:00<?, ?it/s]

acl | epoch 1/10 | train loss 1.1675 | val loss 2.1031 | F1 0.2857 | AUC 0.8419


  0%|          | 0/1129 [00:00<?, ?it/s]

  0%|          | 0/119 [00:00<?, ?it/s]

acl | epoch 2/10 | train loss 1.0136 | val loss 1.1486 | F1 0.7434 | AUC 0.8644


... (23 outputs left out)

acl | epoch 10/10 | train loss 0.6478 | val loss 0.9356 | F1 0.7759 | AUC 0.8812


  0%|          | 0/119 [00:00<?, ?it/s]

\nBest acl checkpoint metrics at threshold 0.5:


,0
accuracy,0.781513
precision,0.725806
sensitivity,0.833333
specificity,0.738462
f1,0.775862
TP,45.000000
FP,17.000000
TN,48.000000
FN,9.000000
roc_auc,0.881197


## 13. Train meniscus model

A fresh three-plane model is initialized for meniscal tear detection.


In [ ]:
meniscus_model, meniscus_history, meniscus_predictions, meniscus_metrics = train_task(MENISCUS_COL, "meniscus")


## 14. Compare and save all three task results


In [ ]:
summary = pd.DataFrame([
    {"task": "Abnormality", **abnormal_metrics},
    {"task": "ACL", **acl_metrics},
    {"task": "Meniscus", **meniscus_metrics},
])
display(summary)

for task_name, history in [
    ("Abnormality", abnormal_history),
    ("ACL", acl_history),
    ("Meniscus", meniscus_history),
]:
    print(f"\n{task_name} training history")
    display(history)

if SAVE_CHECKPOINTS:
    abnormal_predictions.to_csv(CHECKPOINT_DIR / "abnormal_validation_predictions.csv", index=False)
    acl_predictions.to_csv(CHECKPOINT_DIR / "acl_validation_predictions.csv", index=False)
    meniscus_predictions.to_csv(CHECKPOINT_DIR / "meniscus_validation_predictions.csv", index=False)
    summary.to_csv(CHECKPOINT_DIR / "summary_metrics.csv", index=False)
    print("Saved checkpoints, predictions and summary to:", CHECKPOINT_DIR)


## 15. Colab troubleshooting and next steps

If CUDA runs out of memory:
1. Keep `BATCH_SIZE = 1`.
2. Reduce `SLICE_BATCH_SIZE` from 16 to 8 or 4.
3. If doing a smoke test only, set `MAX_SLICES_PER_VIEW = 24` (do not report smoke-test metrics as final results).
4. Restart the Colab runtime if memory remains occupied.

If training is too slow:
- First time one epoch and estimate total runtime.
- Keep all slices for the final run if feasible; subsampling changes the experiment.
- Do not silently compare a slice-subsampled run with the full-slice runs from your previous experiments.

**Interpretation:** Compare our results with the paper cautiously. Their reported validation set contains 120 cases, and their data split, preprocessing, implementation details and model initialization may differ from ours. Our implementation is inspired by their three-plane feature-fusion design, not a bit-for-bit reproduction.